# Chapitre 1.3 : le tokenizer

Une IA ne lit pas des lettres, elle lit des nombres. Avant tout le reste, un outil découpe le texte en morceaux, les **jetons**, et donne un numéro à chaque morceau. Cet outil s’appelle un *tokenizer*. Ici, tu fais tourner celui du chapitre, écrit en Python.

Ce carnet accompagne [la page du chapitre 1.3](https://chipat-neko.github.io/forge-ia/partie-1/1-3/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. récupérer le code du projet ;
2. faire un tour de l’algorithme BPE à la main ;
3. apprendre toutes les fusions sur un petit texte ;
4. découper une phrase avec 0, 30 puis 60 fusions ;
5. essayer tes propres phrases, puis relever un défi ;
6. vérifier que ce carnet trouve exactement les mêmes résultats que la page du site.

**Comment l’utiliser**

Il n’y a rien à installer. Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Tu peux modifier le code et le relancer autant de fois que tu veux : tu ne peux rien casser.

## Étape 0 : récupérer le code

Le code du tokenizer est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, installe la petite bibliothèque `regex` dont il a besoin, puis indique à Python où trouver le projet. Dans Colab, cela prend quelques secondes.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger ni installer.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia
    !pip install --quiet regex

projet = depot / "projets" / "1.3-tokenizer"
if not projet.exists():
    raise FileNotFoundError("Le dépôt n’a pas pu être téléchargé. Vérifie ta connexion, puis relance cette cellule.")

# Python cherche les modules dans les dossiers de sys.path : on y ajoute le projet.
sys.path.insert(0, str(projet.resolve()))
print("Le code du projet est prêt.")

## Étape 1 : un tour de BPE, à la main

BPE veut dire *Byte Pair Encoding*, « encodage par paires ». L’idée tient en quatre phrases :

1. on découpe un texte d’apprentissage en mots, puis chaque mot en lettres ;
2. on compte combien de fois chaque paire de symboles voisins apparaît ;
3. on colle la paire la plus fréquente en un nouveau symbole : c’est une **fusion** ;
4. on recommence, tant qu’une paire apparaît au moins deux fois.

La cellule suivante charge le texte d’apprentissage, un petit texte français d’une quinzaine de lignes, et fait le premier tour. La fonction `visible` rend les espaces lisibles : `␣` est une espace.

In [ ]:
from tokenizer_pedagogique import TokenizerPedagogique, charger_corpus
from tokenizer_pedagogique.bpe import compter_mots, compter_paires, fusionner, paire_la_plus_frequente


def visible(texte):
    # Une espace ne se voit pas : on l’écrit « ␣ » (et un retour à la ligne « ↵ »).
    return texte.replace(" ", "␣").replace("\n", "↵")


def citer(texte):
    # Entre guillemets français. \u202f est l’espace fine insécable de la typographie française.
    return "«\u202f" + visible(texte) + "\u202f»"


corpus = charger_corpus()
print(f"Le texte d’apprentissage compte {len(corpus)} caractères.")

# Chaque mot est écrit lettre par lettre, avec son nombre d’apparitions.
mots = compter_mots(corpus)
print(f"Il contient {len(mots)} mots différents (l’espace qui précède un mot reste collée à lui).")

# On compte les paires de symboles voisins.
paires = compter_paires(mots)
print("Les trois paires les plus fréquentes au départ.")
for (gauche, droite), nombre in paires.most_common(3):
    print(f"  {citer(gauche)} + {citer(droite)}, vue {nombre} fois")

# On colle la plus fréquente : c’est la première fusion.
gagnante = paire_la_plus_frequente(paires)
print(f"On colle donc {citer(gagnante[0])} et {citer(gagnante[1])} en un seul symbole, {citer(gagnante[0] + gagnante[1])}.")

# Exemple sur le mot « les », avec l’espace qui le précède.
avant = tuple(" les")
apres = next(iter(fusionner({avant: 1}, gagnante)))
print("Avant la fusion, ce mot s’écrit", [visible(s) for s in avant])
print("Après la fusion, il s’écrit", [visible(s) for s in apres])

## Étape 2 : apprendre toutes les fusions

Le tokenizer répète ce tour jusqu’à ce qu’aucune paire n’apparaisse au moins deux fois. Chaque fusion est numérotée dans l’ordre où elle a été apprise : c’est ce même ordre qu’il rejoue pour découper un nouveau texte.

In [ ]:
tokenizer = TokenizerPedagogique(corpus)

print(f"{len(tokenizer.fusions)} fusions apprises, à partir de {len(tokenizer.base)} caractères de base.")
print("Les dix premières, dans l’ordre où elles ont été apprises.")
for numero, fusion in enumerate(tokenizer.fusions[:10], start=1):
    print(f"{numero:>3}. {citer(fusion.a)} + {citer(fusion.b)} donnent {citer(fusion.a + fusion.b)}, vue {fusion.compte} fois")

## Étape 3 : découper une phrase

Pour découper un texte, le tokenizer le coupe d’abord en lettres, puis rejoue les `k` premières fusions, dans l’ordre. Avec `k = 0`, chaque lettre est un jeton. Plus `k` grandit, plus les morceaux fréquents se collent, et plus la suite de jetons raccourcit.

C’est le nombre que tu fais glisser dans l’atelier de la page. Ici, la phrase est découpée avec 0, 30, puis 60 fusions. Les barres verticales séparent les jetons.

In [ ]:
phrase = "Les forgerons découpent le texte en jetons."

nombres = {}
for k in (0, 30, 60):
    jetons = tokenizer.decouper(phrase, k)
    nombres[k] = len(jetons)
    mot_fusion = "fusion" if k < 2 else "fusions"
    print(f"Avec {k} {mot_fusion}, {len(jetons)} jetons.")
    print("   " + " | ".join(visible(jeton.texte) for jeton in jetons))

print()
print(f"La même phrase passe de {nombres[0]} jetons à {nombres[60]} jetons.")

## Étape 4 : ce que reçoit vraiment l’IA

L’IA ne reçoit pas les morceaux de texte, mais leurs **numéros**. Chaque caractère du texte d’apprentissage a un numéro : sa place dans la liste triée des caractères. Chaque fusion crée un nouveau jeton, numéroté à la suite des caractères : la première fusion prend le premier numéro libre, la deuxième le suivant, et ainsi de suite.

In [ ]:
jetons = tokenizer.decouper(phrase, 60)
for jeton in jetons:
    print(f"{citer(jeton.texte):<12} numéro {jeton.id}")

print()
print("Pour le réseau de neurones, la phrase se réduit à cette suite de nombres.")
print([jeton.id for jeton in jetons])

## Étape 5 : à toi de jouer

Change la phrase et le nombre de fusions `k`, puis relance la cellule. Au départ, elle reprend la phrase et le réglage de l’atelier de la page, avec 20 fusions.

Essaie des mots du texte d’apprentissage (« forgeron », « jetons », « modèle »), puis des mots rares (« reforgeronnes »). Essaie aussi un caractère que le texte n’a jamais vu, comme un emoji ou une lettre étrangère : il n’a pas de numéro, et le carnet écrit `inconnu` à sa place.

In [ ]:
ma_phrase = "Le forgeron frappe le métal sur l'enclume."
k = 20  # le nombre de fusions rejouées, de 0 à len(tokenizer.fusions)

jetons = tokenizer.decouper(ma_phrase, k)
print(f"{len(jetons)} jetons pour {len(ma_phrase)} caractères.")
print(" | ".join(visible(jeton.texte) for jeton in jetons))
print(" | ".join("inconnu" if jeton.id is None else str(jeton.id) for jeton in jetons))

## Exercice : le défi de la page

Sur la page du chapitre, un défi te demande une phrase de **30 caractères ou plus** qui tient en **12 jetons ou moins** avec toutes les fusions apprises. Relève-le ici : écris ta phrase dans la cellule suivante, puis lance-la pour savoir si le défi est réussi.

Indice : les mots que le texte d’apprentissage contient souvent sont déjà devenus des jetons entiers. Il suffit d’en enchaîner quelques-uns.

In [ ]:
ma_phrase = "Écris ta phrase ici."  # à remplacer par ta phrase

jetons = tokenizer.decouper(ma_phrase, len(tokenizer.fusions))
print(f"Ta phrase a {len(ma_phrase)} caractères et tient en {len(jetons)} jetons.")
if len(ma_phrase) >= 30 and len(jetons) <= 12:
    print("Défi réussi, bravo.")
else:
    print("Pas encore. Il faut au moins 30 caractères et au plus 12 jetons.")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

Une solution parmi d’autres, qui fait 41 caractères et tient en 9 jetons seulement : la plupart de ses mots sont assez fréquents dans le texte d’apprentissage pour être devenus des jetons entiers, avec l’espace qui les précède.

Remplace la première ligne de la cellule de l’exercice par :

```python
ma_phrase = "Le modèle découpe les jetons en morceaux."
```

À l’inverse, une phrase pleine de mots rares se casse en beaucoup de morceaux : essaie « reforgeronnes » dans la cellule de l’étape 5.

</details>

## Dernière étape : le même résultat que la page ?

La page du site n’utilise pas Python : elle refait le même calcul en TypeScript, directement dans ton navigateur. Pour être sûr que les deux donnent exactement les mêmes jetons, le dépôt garde un fichier de référence, `tests/attendus.json`, calculé une fois pour toutes : les fusions apprises, puis le découpage de plusieurs textes avec plusieurs nombres de fusions. Les tests de la page relisent ce fichier. La cellule suivante le relit aussi, et le compare à ce que ton tokenizer vient de calculer.

In [ ]:
import json

attendus = json.loads((projet / "tests" / "attendus.json").read_text(encoding="utf-8"))

# Les fusions apprises ici sont-elles celles du fichier de référence ?
memes_fusions = [[f.a, f.b, f.compte] for f in tokenizer.fusions] == attendus["fusions"]

# Chaque découpage du fichier est-il retrouvé, jeton par jeton, numéro par numéro ?
cas_differents = []
for cas in attendus["cas"]:
    jetons = [[j.texte, j.id, j.mot] for j in tokenizer.decouper(cas["texte"], cas["k"])]
    if jetons != cas["jetons"]:
        cas_differents.append(cas)

print(f"{len(tokenizer.fusions)} fusions et {len(attendus['cas'])} découpages comparés.")
if not memes_fusions:
    print("Les fusions apprises ne sont pas celles du fichier de référence.")
for cas in cas_differents[:3]:
    print(f"Découpage différent pour {citer(cas['texte'])} avec {cas['k']} fusions.")

identique = memes_fusions and not cas_differents
print(f"Même résultat que la page du site\u202f: {'oui' if identique else 'non'}")  # \u202f est l’espace fine insécable

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/1.3-tokenizer sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/1.3-tokenizer).
- La page du chapitre, pour retrouver l’atelier, les paris et le code expliqué : [chapitre 1.3](https://chipat-neko.github.io/forge-ia/partie-1/1-3/).
- Le tokenizer des grands modèles travaille sur des octets plutôt que sur des lettres, mais le principe est exactement celui que tu viens de manipuler.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.